# TOFU True Retraining MIAU Experiment

This notebook runs the TOFU MIAU experiment with **true retraining** on Google Colab.

**Key features:**
- Uses base Phi-1.5 (not pre-trained on TOFU)
- True retraining from scratch
- 1000 samples, 10 seeds, forget set differs per seed
- Faithful DualOptim+ (`AdamWDecouplePlus` optimizer, ME+GD alternation)
- Balanced membership-inference evaluation with cross-validation
- Checkpoint saving/resumption
- Results saved to Google Drive

## Before you run

Upload **`tofu_retrain_experiment.py`** to the same Drive folder as this notebook.
All experiment logic lives there; this notebook only mounts Drive, fetches data,
and calls into it. Keeping one copy of the code avoids the two drifting apart.

**Estimated time:** 3-6 hours on A100

> **Memory:** DualOptim+ keeps six optimizer state tensors per parameter (a shared
> base pair plus a delta pair for each objective). In fp32 that needs roughly 45 GB
> for Phi-1.5, which will not fit on a 40 GB A100. If you are not on an 80 GB card,
> set `optimizer_state_dtype='bfloat16'` in the config cell. The run logs an
> estimate and warns before training starts.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ============================================================
# CONFIGURE THIS: Set to the Drive folder containing this notebook
# Results will be saved in the SAME folder as the notebook
# ============================================================
NOTEBOOK_FOLDER = '/content/drive/MyDrive/Colab Notebooks'  # <-- CHANGE THIS to your notebook location

# Results and checkpoints will be saved directly to Drive (same folder as notebook).
# The checkpoint folder is versioned: models trained before the prompt-template,
# sequence-length and retain-split changes are not compatible with the current code,
# and load_or_create_model only checks whether a file exists.
RESULTS_DIR = os.path.join(NOTEBOOK_FOLDER, 'TOFU_Phi1.5_TrueRetrain')
CHECKPOINT_DIR = os.path.join(NOTEBOOK_FOLDER, 'checkpoints_tofu_v2')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Results will be saved to: {RESULTS_DIR}")
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## 2. Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Install Dependencies

In [ ]:
!pip install -q transformers accelerate datasets scikit-learn tqdm

## 4. Download TOFU Dataset

In [ ]:
import json
import os
from datasets import load_dataset

os.makedirs("data/tofu", exist_ok=True)

# Use the full TOFU corpus (4000 questions) and partition it ourselves:
# - First 2000 rows → forget pool (sample 400 per seed, ~20% overlap between seeds)
# - Last 2000 rows → retain pool (split into 1000/1000 halves for retain/test)
#
# This gives meaningful forget-set variance across seeds, unlike using HF's
# forget10 (400 rows) where every seed would train on identical data.

N_FORGET, N_RETAIN, N_TEST = 400, 1000, 1000

print("Downloading TOFU full dataset...")

ds = load_dataset("locuslab/TOFU", "full", split="train")
all_data = [{"question": x["question"], "answer": x["answer"]} for x in ds]

print(f"Full dataset: {len(all_data)} rows")

# Partition: first half for forget, second half for retain/test
midpoint = len(all_data) // 2
forget_pool = all_data[:midpoint]
retain_pool = all_data[midpoint:]

print(f"Forget pool: {len(forget_pool)} rows")
print(f"Retain pool: {len(retain_pool)} rows")

# Fail loudly rather than silently degrading the experiment.
if len(forget_pool) < N_FORGET:
    raise ValueError(
        f"Forget pool has {len(forget_pool)} rows but {N_FORGET} are needed per seed. "
        "With too few rows every seed would get an identical forget set."
    )
if len(retain_pool) < 2 * max(N_RETAIN, N_TEST):
    raise ValueError(
        f"Retain pool has {len(retain_pool)} rows, too few to draw {N_RETAIN} retain "
        f"and {N_TEST} test records from two disjoint halves."
    )

key = lambda rows: {(r["question"], r["answer"]) for r in rows}
overlap = key(forget_pool) & key(retain_pool)
if overlap:
    raise ValueError(f"Forget and retain pools share {len(overlap)} records; they must be disjoint.")

with open("data/tofu/forget_pool.json", "w") as f:
    json.dump(forget_pool, f, indent=2)
with open("data/tofu/retain_pool.json", "w") as f:
    json.dump(retain_pool, f, indent=2)

print("\nTOFU data downloaded and validated successfully!")
print(f"Cross-seed forget overlap: ~{N_FORGET**2 // len(forget_pool)}/{N_FORGET} ({100*N_FORGET//len(forget_pool)}%)")

## 5. Load Experiment Code

The experiment code lives in `tofu_retrain_experiment.py`, which must be in the
same Drive folder as this notebook. It provides:
- True retraining from base Phi-1.5
- Graded retraining (25%, 50%, 75%, 100%), nested so each level is a subset of the last
- DualOptim+ unlearning via the ported `AdamWDecouplePlus` optimizer
- Balanced MIA and MIAU computation
- Checkpoint saving/resumption

In [ ]:
import importlib
import os
import sys

SCRIPT_NAME = 'tofu_retrain_experiment.py'
script_path = os.path.join(NOTEBOOK_FOLDER, SCRIPT_NAME)

if not os.path.exists(script_path):
    raise FileNotFoundError(
        f"Could not find {SCRIPT_NAME} at {script_path}.\n"
        f"Upload it to the same Drive folder as this notebook, then re-run this cell."
    )

if NOTEBOOK_FOLDER not in sys.path:
    sys.path.insert(0, NOTEBOOK_FOLDER)

import tofu_retrain_experiment as exp
importlib.reload(exp)  # pick up edits without restarting the runtime

print(f"Loaded experiment code from: {script_path}")
print(f"DualOptim+ optimizer: {exp.AdamWDecouplePlus.__name__}")



## 6. Run Experiment

In [ ]:
config = exp.TOFURetrainConfig(
    # Data: 5 seeds with completely disjoint forget sets (400 × 5 = 2000 pool).
    # Retain and test from two disjoint 1000-row halves of the other 2000.
    data_path='data/tofu',
    split='forget_pool',
    retain_source='retain_pool',
    max_forget_samples=400,
    max_retain_samples=1000,
    max_test_samples=1000,
    seeds=[1, 2, 3, 4, 5],
    disjoint_forget=True,

    # Baseline / graded retraining
    num_epochs=3,
    batch_size=4,

    # DualOptim+ defaults already match scripts/tofu_phi1-5/me_gd_dual.sh
    # (forget_coeff=1.0, retain_freq=5, betas=(0.9, 0.95, 0.9, 0.95), max_steps=300).
    # On a 40 GB A100 the six fp32 optimizer state tensors will not fit; uncomment:
    # optimizer_state_dtype='bfloat16',

    device='cuda',
    results_dir=RESULTS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
)

print("Starting experiment...")
print(f"Total samples: {config.max_forget_samples + config.max_retain_samples + config.max_test_samples}")
print(f"Seeds: {config.seeds}")
print(f"Device: {config.device}")
print(f"Results: {config.results_dir}")

results = exp.run_experiment(config)

## 7. Verify Results Location

In [ ]:
# Results are saved DIRECTLY to Drive (no copying needed)
# They are in the same folder as this notebook

print("Results saved directly to Google Drive:")
print(f"  Results: {RESULTS_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print()

# List files in results directory
if os.path.exists(RESULTS_DIR):
    files = os.listdir(RESULTS_DIR)
    print(f"Files in results folder ({len(files)} files):")
    for f in sorted(files):
        print(f"  - {f}")
else:
    print("Results folder not found yet. Run the experiment first.")

## 8. View Summary

In [ ]:
# Display summary table from Drive
summary_file = os.path.join(RESULTS_DIR, 'summary_table.txt')
print(f'Looking for: {summary_file}')
print()

if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        print(f.read())
else:
    print('Summary table not found. Run the experiment first.')